<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. 데이터 불러오기
try:
    df = pd.read_spss("/content/HN24_ALL.sav")
except ImportError:
    import os
    os.system('pip install pyreadstat')
    df = pd.read_spss("/content/HN24_ALL.sav")

# 변수명 매핑
sex_col = 'sex'
age_col = 'age'
dm_col = 'HE_DM_HbA1c'
hp_col = 'HE_HP'
ob_col = 'HE_obe'
incm_col = 'incm'
edu_col = 'edu'
bs_col = 'BS3_1'
bd_col = 'BD1_11'

covariates = [sex_col, ob_col, incm_col, edu_col, bs_col, bd_col]

# ---------------------------------------------------------
# [사전 확인] 주요 변수 빈도표 (결측 및 8, 9 제외 기준)
# ---------------------------------------------------------
print("=== [사전 확인] 주요 변수 빈도표 (결측 및 8, 9 제외) ===")
for var in check_vars:
    print(f"\n▶ 변수명: [{var}]")
    temp_series = pd.to_numeric(df[var], errors='coerce')

    if var in [bs_col, bd_col]:
        valid_mask = temp_series.notnull() & (temp_series != 9)
    else:
        valid_mask = temp_series.notnull() & (~temp_series.isin([8, 9]))

    freq_table = temp_series[valid_mask].value_counts().sort_index()
    freq_df = pd.DataFrame({
        '코드(응답)': freq_table.index.astype(int),
        '빈도수(N)': freq_table.values
    })
    display(freq_df)

# ---------------------------------------------------------
# [분석 대상자 선정 플로우차트 (5,458명 고정)]
# ---------------------------------------------------------
print("=== [데이터 정제] 분석 대상자 선정 단계 ===")
print(f"Step 0. 전체 조사 대상자: N = {len(df):,}")

# 1단계. 만 19세 이상 성인 포함
df[age_col] = pd.to_numeric(df[age_col], errors='coerce')
df_step1 = df[df[age_col] >= 19].copy()
print(f"Step 1. 만 19세 이상 성인 포함: N = {len(df_step1):,} (제외: {len(df) - len(df_step1):,}명)")

# 2단계. 당뇨병 유병 여부 확인 가능 대상자
df_step1[dm_col] = pd.to_numeric(df_step1[dm_col], errors='coerce')
df_step2 = df_step1[df_step1[dm_col].notnull() & (~df_step1[dm_col].isin([8, 9]))].copy()
print(f"Step 2. 당뇨병 유병 여부 확인 가능 (노출변수 결측/8/9 제외): N = {len(df_step2):,} (제외: {len(df_step1) - len(df_step2):,}명)")

# 3단계. 고혈압 유병 여부 확인 가능 대상자 (Table 1 기본 대상자: 5,458명)
df_step2[hp_col] = pd.to_numeric(df_step2[hp_col], errors='coerce')
df_step3 = df_step2[df_step2[hp_col].notnull() & (~df_step2[hp_col].isin([8, 9]))].copy()
print(f"Step 3. 고혈압 유병 여부 확인 가능 (결과변수 결측/8/9 제외): N = {len(df_step3):,} (제외: {len(df_step2) - len(df_step3):,}명)")

# 기본 분석 대상자 고정 (5,458명)
df_base = df_step3.copy()
print("-" * 50)
print(f">> 최종 기본 분석 대상자 수 (Table 1 기본 대상): N = {len(df_base):,}명 (유지)")
print("-" * 50)

# ---------------------------------------------------------
# [공변량별 결측 및 비해당/무응답 빈도 확인]
# ---------------------------------------------------------
print("\n=== [4단계 공변량별 결측 빈도 및 제외 인원 분석] ===")
missing_report = []

for col in covariates:
    df_base[col] = pd.to_numeric(df_base[col], errors='coerce')

    if col in [bs_col, bd_col]:
        # 흡연, 음주는 8(비해당)을 정상으로 인정하고, 결측(NaN) 및 9(모름)만 결측 처리
        missing_mask = df_base[col].isnull() | (df_base[col] == 9)
    else:
        # 성별, 비만, 소득, 교육은 결측(NaN), 8(비해당), 9(모름) 모두 결측 처리
        missing_mask = df_base[col].isnull() | df_base[col].isin([8, 9])

    missing_n = missing_mask.sum()
    valid_n = len(df_base) - missing_n

    missing_report.append({
        '공변량 변수명': col,
        '유효 데이터(N)': valid_n,
        '결측/제외 대상(N)': missing_n
    })

missing_df = pd.DataFrame(missing_report)
display(missing_df)

# ---------------------------------------------------------
# [참고] 5단계. 교란변수 결측 모두 제외 시 최종 회귀분석 대상자 (5,300명 확인용)
# ---------------------------------------------------------
df_regression = df_base.copy()
for col in covariates:
    if col in [bs_col, bd_col]:
        df_regression = df_regression[df_regression[col].notnull() & (df_regression[col] != 9)]
    else:
        df_regression = df_regression[df_regression[col].notnull() & (~df_regression[col].isin([8, 9]))]

print("-" * 50)
print(f">> [참고] 교란변수 결측 전체 제외 시 최종 회귀분석 대상자: N = {len(df_regression):,}명")
print("-" * 50)

=== [사전 확인] 주요 변수 빈도표 (결측 및 8, 9 제외) ===

▶ 변수명: [sex]


,코드(응답),빈도수(N)
0,1,3070
1,2,3927



▶ 변수명: [age]


,코드(응답),빈도수(N)
0,1,30
1,2,30
2,3,40
3,4,38
4,5,37
...,...,...
73,76,105
74,77,81
75,78,55
76,79,55



▶ 변수명: [HE_DM_HbA1c]


,코드(응답),빈도수(N)
0,1,2813
1,2,1844
2,3,959


... (11 outputs left out)

,코드(응답),빈도수(N)
0,1,1115
1,2,1179
2,3,671
3,4,1219
4,5,800
5,6,292
6,8,1496


=== [데이터 정제] 분석 대상자 선정 단계 ===
Step 0. 전체 조사 대상자: N = 6,997
Step 1. 만 19세 이상 성인 포함: N = 6,033 (제외: 964명)
Step 2. 당뇨병 유병 여부 확인 가능 (노출변수 결측/8/9 제외): N = 5,616 (제외: 417명)
Step 3. 고혈압 유병 여부 확인 가능 (결과변수 결측/8/9 제외): N = 5,567 (제외: 49명)
--------------------------------------------------
>> 최종 기본 분석 대상자 수 (Table 1 기본 대상): N = 5,567명 (유지)
--------------------------------------------------

=== [4단계 공변량별 결측 빈도 및 제외 인원 분석] ===


,공변량 변수명,유효 데이터(N),결측/제외 대상(N)
0,sex,5567,0
1,HE_obe,5495,72
2,incm,5547,20
3,edu,5442,125
4,BS3_1,5447,120
5,BD1_11,5448,119


--------------------------------------------------
>> [참고] 교란변수 결측 전체 제외 시 최종 회귀분석 대상자: N = 5,367명
--------------------------------------------------


In [ ]:
# ---------------------------------------------------------
# [사전 분석] 변수 및 복합표본설계 변수, 원 코드 빈도 확인
# ---------------------------------------------------------

# 1. 복합표본설계 변수 (층, 집락, 가중치) 확인
# SPSS 원시자료의 대소문자 차이를 방지하기 위해 일치하는 컬럼을 찾습니다.
target_design_vars = ['kstrata', 'psu', 'wt_itvex']
design_cols = [col for col in df.columns if col.lower() in target_design_vars]

print("=== 1. 복합표본설계 변수 확인 ===")
if len(design_cols) == 3:
    print(f"분석에 필요한 설계 변수가 모두 존재합니다: {design_cols}")
    # 가중치의 요약 통계량(최소, 최대, 평균 등)을 확인하여 이상치 여부 점검
    display(df[design_cols].describe())
else:
    print(f"경고: 일부 설계 변수를 찾을 수 없습니다. 현재 확인된 변수: {design_cols}")

# 2. 분석 주요 변수 리스트 (앞서 정의한 매핑 변수 사용)
check_vars = [age_col, dm_col, hp_col, ob_col, incm_col, edu_col, bs_col, bd_col]

=== 1. 복합표본설계 변수 확인 ===
분석에 필요한 설계 변수가 모두 존재합니다: ['psu', 'wt_itvex', 'kstrata']


,wt_itvex,kstrata
count,6997.000000,6997.000000
mean,7363.446477,111.520509
std,3807.130476,7.353178
min,314.853524,101.000000
25%,4738.510601,106.000000
50%,6665.953013,110.000000
75%,9024.211026,117.000000
max,30575.037524,127.000000
